In [2]:
import time
from pathlib import Path
from eproc_driver import eproc as eproc
import os
import re
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys

import configparser
import pyautogui
import sqlite3
import pyperclip

config = configparser.ConfigParser()
config.read("usuario.txt")
            
# path to config file
LOGIN = config.get("vars", "LOGIN")
SENHA = config.get("vars", "SENHA")
DOWNLOADPATH = config.get("vars", "DOWNLOADPATH")
print("Configurações do usuário importadas.")


# inicializa tudo,  cria um browser Chrome
options=eproc.configura_webdriver(DOWNLOADPATH)
browser=eproc.novo_browser(options)
driver=eproc.novo_webdriver()


Configurações do usuário importadas.


In [ ]:
import pyotp
pyotop_code = "GJRGIYTCGBSGKYTEHE2TOZRUGFQTMMRQ"
totp = pyotp.TOTP(pyotop_code)
totp.now()

'847213'

In [2]:
def trataMinuta(texto, tipo_ato):
    # Regex para capturar o texto a partir do tipo_ato até @NUMEROPROCESSOFORMATADO@
    # O re.escape é usado para escapar caracteres especiais no tipo_ato
    padrao = re.compile(
        rf"{re.escape(tipo_ato)}.*?(@NUMEROPROCESSOFORMATADO@)", 
        re.IGNORECASE | re.DOTALL | re.UNICODE
    )    
    match = padrao.search(texto)    
    if not match:
        return None  # Retorna None se não encontrou o padrão    
    texto_limpo = match.group(0)    
    # Remove tudo depois de @NUMEROPROCESSOFORMATADO@ (inclusive o que vier depois dele)
    texto_limpo = re.sub(r"(@NUMEROPROCESSOFORMATADO@).*", r"\1", texto_limpo, flags=re.DOTALL)    
    # Remove quebras de linha em excesso (mais de 2 quebras viram 2)
    texto_limpo = re.sub(r'\n\s*\n+', '\n\n', texto_limpo)    
    # Remove espaços em excesso nas linhas
    texto_limpo = '\n'.join(linha.strip() for linha in texto_limpo.splitlines())    
    return texto_limpo

def pegaMinuta(driver, cod_minuta):
    pyautogui.click(1000, 600)
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").clear()
    time.sleep(0.2)  # Pequena pausa para segurança
    #insere o código 
    driver.find_element(By.ID, "txtCodigoModelo").click()
    time.sleep(0.2)  # Pequena pausa para segurança
    driver.find_element(By.ID, "txtCodigoModelo").send_keys(cod_minuta)
    time.sleep(2)
    pyautogui.hotkey('enter')
    time.sleep(4)
    # Localiza o elemento com código
    elemento = driver.find_element(By.PARTIAL_LINK_TEXT, str(cod_minuta))
    # Cria uma cadeia de ações e move o mouse até o elemento
    actions = ActionChains(driver)
    actions.move_to_element(elemento).perform()
    time.sleep(4)
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()
    pyautogui.click(1000, 600)
    pyautogui.hotkey('f5')
    time.sleep(3)  # Pequena pausa para segurança
    #devolve o conteudo
    return conteudo

def gravaTextoMinuta(cod_minuta, texto_minuta):
    conn = sqlite3.connect('minutas.db')
    cursor = conn.cursor()
    query_insert = 'UPDATE minutas SET conteudo = ? WHERE Código = ?;'
    cursor.execute(query_insert, (texto_minuta, cod_minuta))
    conn.commit()
    conn.close()

def pegaProximaMinutaVazia():
    conn = sqlite3.connect('minutas.db')
    cursor = conn.cursor()
    query_select='SELECT "Código", "Tipo de Documento" FROM minutas WHERE conteudo IS NULL LIMIT 1'
    cursor.execute(query_select)
    resultados = cursor.fetchall()
    return resultados

In [3]:
#EXECUTA!
conn = sqlite3.connect('minutas.db')
cursor = conn.cursor()
query_select = 'SELECT COUNT(*) FROM minutas WHERE conteudo IS NULL'
cursor.execute(query_select)
resultado = cursor.fetchone()[0]  # pega o número da tupla (ex: (42,) -> 42)
for _ in range(resultado):
    resultados = pegaProximaMinutaVazia()
    for r in resultados:
        try:
            cod_minuta = r[0]
            texto_minuta = pegaMinuta(driver, cod_minuta)
            gravaTextoMinuta(cod_minuta, texto_minuta)  
            print(f"Texto da minuta {cod_minuta} capturado")  
        except:
            print("Erro. Passando para a próxima")


            print("teste")

Texto da minuta 10000362478 capturado
Texto da minuta 10000286060 capturado
Texto da minuta 10001548485 capturado
Texto da minuta 10001568031 capturado
Texto da minuta 10001493192 capturado
Texto da minuta 10001383528 capturado
Texto da minuta 10001419597 capturado
Texto da minuta 10001386384 capturado
Texto da minuta 10001428850 capturado
Texto da minuta 10001395626 capturado
Texto da minuta 10000584214 capturado
Texto da minuta 10001566697 capturado
Texto da minuta 10001566686 capturado
Texto da minuta 10001545915 capturado
Texto da minuta 10001546998 capturado
Texto da minuta 10001546306 capturado
Texto da minuta 10001566671 capturado
Texto da minuta 10001563240 capturado
Texto da minuta 10001566670 capturado
Texto da minuta 10001566705 capturado
Texto da minuta 10001550964 capturado
Texto da minuta 10001104807 capturado
Texto da minuta 10001394309 capturado
Texto da minuta 10001556295 capturado
Texto da minuta 10001567356 capturado
Texto da minuta 10001045719 capturado
Texto da min

In [ ]:
conn.close()
